# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. The approach is fully referenced by Croissant schema `@id` fields for reproducibility and robust programmatic access.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

Every *record set*, *field*, and *column* is referenced via its `@id`.

*Note*: The FAIR^2 Croissant schema groups tabular data into one or more record sets. Let's list all available record sets and enumerate their fields using their `@id` fields.

In [ ]:
# List all record sets and their field and column IDs
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rset in record_sets:
    print(f"Record set name: {rset.name} (ID: {rset.id})")
    print("  Data file object ID:", getattr(rset, 'data', 'N/A'))
    print("  Fields and columns:")
    for fld in rset.fields:
        col_id = getattr(fld, 'column', None)
        print(f"    Field: {fld.id}  Column: {col_id if col_id else 'N/A'}")
    print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the record set and field `@id` references discovered above.

Note: If multiple record sets are present, you may extract each one into a separate DataFrame.

In [ ]:
# Extract data from each record set, using @id for dynamic referencing
dataframes = {}
ids_to_print = []  # Will be filled with record set @id(s)

for rset in dataset.record_sets:
    recset_id = rset.id
    try:
        recs = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(recs)
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} records for Record Set @id: {recset_id}")
        ids_to_print.append(recset_id)
        print("Columns (using field @id names):", df.columns.tolist())
    except Exception as e:
        print(f"Failed to load record set {recset_id}: {e}")

# Preview the first table
if ids_to_print:
    display(dataframes[ids_to_print[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We will:
* Select a numeric field by its field `@id`
* Filter for high values
* Normalize values
* Optionally group by a categorical field

> Replace `your_record_set_id`, `your_numeric_field_id`, `your_group_field_id` with IDs from the actual schema printed above or use those present (demonstrated with an example if possible)


In [ ]:
# ----- Configuration: specify relevant @ids here based on printed lists above -----
# For demonstration, we attempt to automatically pick a numeric field if present, else user should inspect (see printout above).

record_set_id = ids_to_print[0] if ids_to_print else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# Heuristic to pick a likely numeric field
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

# If not found, try common field names
if not numeric_field_id:
    for candidate in ['cr:coveragePercent', 'cr:peptideCount', 'cr:molWeight', 'cr:pI', 'cr:abundance']:
        if candidate in df.columns:
            numeric_field_id = candidate
            break

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if len(df)>0 else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > mean (mean={threshold:.2f}): {len(filtered_df)} records")

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records (first 5 rows):")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    # Try grouping by the first non-numeric field
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field} (first 5 groups):")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA. Please examine above and set 'numeric_field_id' manually to a field '@id' containing numeric data.")

## 5. Visualization
Visualize data distributions or relationships between numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If grouping field exists, boxplot
    if group_field:
        n_groups = filtered_df[group_field].nunique()
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        if n_groups > 8:
            plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion

* This walkthrough demonstrated loading a molecular dataset's Croissant schema using `mlcroissant`.
* All data references were made using official `@id` values for clarity and reproducibility.
* The mass spectrometry data was loaded, filtered, normalized, grouped, and visualized with Python data analysis tools.

For sophisticated biological or clinical analysis, consult the schema and field documentation under each field's `@id` for full context and units.

**Next steps**: Refine the field selection manually based on the full schema printout above, then proceed to advanced statistical analysis or integration with machine learning tools.